## Topic 6: Duplicate Documents — Hash + Near-Duplicate Detection

### Concept, Intuition, Why It Exists

- A real production scenario: `FD_Policy.pdf`, `FD_Policy_Final.pdf`, `FD_Policy_V2.pdf` all sitting in the same source folder — three filenames, but the actual content might be byte-identical in two cases and a reworded near-copy in the third. Ingesting all three blindly means the same fact gets embedded multiple times: wasted storage, wasted embedding cost, and retrieval returning near-identical chunks for one query, crowding out genuinely different relevant content.
- Two fundamentally different kinds of duplication need two fundamentally different detection tools: **exact duplicates** (byte-for-byte identical text, different filename — trivial and cheap to detect), and **near duplicates** (the same fact, reworded — requires understanding meaning, not just bytes, which is exactly what embeddings are for).

### Internal Working

**Stage 1 — Hash (cheap, exact, runs on everything):**
1. Compute a SHA-256 hash of each Document's text.
2. Track which hash was seen first while iterating every Document once.
3. If a hash repeats, that Document is an exact duplicate of the first one seen with that hash — recorded immediately, no embedding needed.
4. Anything that does not match an existing hash is kept for Stage 2.

**Stage 2 — Cosine similarity (expensive, near, runs only on survivors):**
1. Only the Documents that survived Stage 1 get embedded — this ordering is the entire point: embeddings cost real compute/money, so pay for them only on content you haven't already rejected for free.
2. Embed all surviving texts in one batch call, with vectors normalized to unit length.
3. Because vectors are normalized, cosine similarity reduces to a plain dot product — which is why normalization is requested explicitly at encode time rather than computing full cosine similarity separately.
4. Compare every surviving pair; any pair scoring at or above a similarity threshold (e.g. 0.97) is flagged as a near duplicate, with a "already flagged" tracker preventing a document from being reported as a duplicate of more than one earlier document.

### How It's Implemented In This Project

- Tested against the exact scenario this topic opened with: a small test set simulating `FD_Policy.json` / `FD_Policy_Final.json` / `FD_Policy_V2.json` correctly caught the byte-identical pair via hash and the reworded pair via cosine similarity — two different kinds of duplication, two different tools, applied in cost order.

### Real-World Issues, Edge Cases, Debugging, Scaling, Cost

- **Pairwise comparison in Stage 2 scales quadratically** with the number of post-hash-dedup Documents. Fine for hundreds of documents; a real problem at tens of thousands. Production systems use approximate nearest neighbor indexes to find near-duplicate candidates much faster instead of comparing every pair.
- **Threshold sensitivity**: the similarity threshold is a judgment call with real consequences in both directions — too low, and genuinely different content with similar phrasing gets wrongly merged or dropped; too high, and obvious near-duplicates slip through. There is no universally correct threshold; it should be tuned against a labeled sample of true positives/negatives for your actual content, not picked once and forgotten.
- **Which copy survives matters**: flagging duplicates doesn't decide which one to keep. A real ingestion pipeline needs an explicit policy — keep the one from the most recently modified file, keep the one with the higher document version, prefer a trusted source folder — because keeping an outdated policy version over a current one is a correctness bug, not just a storage-efficiency issue.
- **Cost accounting**: hashing is essentially free. Embedding is the real cost driver — this is precisely why the hash stage exists as a gate in front of the embedding stage, not as a parallel independent check.
- **Cross-batch duplicates**: this approach only catches duplicates within the list passed to it in one call. A document ingested today that duplicates one ingested last month won't be caught unless the comparison set includes historical embeddings too — meaning real production dedup needs to compare against the existing vector store, not just the current ingestion batch.

### Design Decisions & Trade-offs

- Cost-ordering the two stages is the central design decision: cheapest and most certain check first, most expensive and fuzziest check last, and only on what survives. Reversing the order would still work correctly but would pay embedding cost on documents you'd have rejected for free, strictly worse with no compensating benefit.
- Greedy first-match assignment vs. clustering: this approach greedily assigns each near-duplicate to the first earlier document it matches above threshold, not necessarily its closest match, and doesn't form true duplicate clusters when more than two documents are mutually similar. A clustering approach is more correct for that case but adds real implementation complexity for a problem that, at small scale, doesn't yet need it.

### Alternatives & When To Use Each

- Hash plus brute-force cosine comparison — small-to-medium document counts, simplicity over scale.
- Hash plus an approximate nearest neighbor index for near-duplicate search — tens of thousands or more documents, where pairwise comparison becomes a real bottleneck.
- Locality-sensitive hashing (MinHash/SimHash) — very large-scale near-duplicate detection where even approximate embedding comparison is too expensive, trading semantic understanding for raw speed.
- Vector-store-native dedup at insert time — production systems that already query the vector store on every ingest, checking for near-neighbors as part of the insert rather than as a separate batch pass.

### Common Mistakes & Production Failures

- Running the expensive embedding-based check on every document regardless of the hash check — silently multiplying embedding cost for zero added benefit on documents the hash check would already have caught for free.
- Picking a near-duplicate threshold once during development on a small test set and never revisiting it as real, messier production data starts flowing through.
- Deduplicating without a policy for which duplicate to keep, then discovering an outdated policy document was the one that survived.
- Forgetting that dedup needs to run against the existing vector store too, not just within a single day's ingestion batch — otherwise duplicates accumulate slowly across many ingestion runs instead of being caught once.

### Lead-Level Interview Questions

**Q: Why run the hash check before the embedding check, rather than doing both independently and combining results?**
A: Cost ordering. Hashing is essentially free; embedding costs real compute/money per document. Running hash first means embeddings are only computed for documents that survived the free check — hash-duplicates are a strict subset of what cosine similarity would also catch, since identical text has similarity 1.0, so there's zero correctness benefit to paying for embeddings on them.

**Q: This dedup approach only compares documents within a single ingestion batch. What's the actual risk in production, and how would you fix it?**
A: Duplicates that span ingestion runs go uncaught, because the comparison set never includes historical data. The fix is comparing new documents against the existing vector store at insert time — a nearest-neighbor query against what's already indexed, not just against the other documents in today's batch.

**Q: How would you choose and validate a near-duplicate similarity threshold for a real system, rather than picking a number because it "felt right"?**
A: Build a small labeled set of true near-duplicate pairs and true non-duplicate-but-similar pairs from real data, compute their similarity scores, and pick a threshold that separates them with acceptable false-positive/false-negative rates for your use case — then monitor in production and revisit as data distributions shift.

**Q: Two documents are flagged as near-duplicates, but only one is current policy and the other is stale. What does the dedup function do about that, and is that a gap?**
A: Nothing — it only flags the pair, it doesn't decide which to keep. That's a real gap for production use: an explicit "which copy wins" policy is required on top of dedup detection; treating "duplicate detected" as automatically solved would silently let a stale, wrong document win in some ordering-dependent way.

### Hidden Concepts Worth Knowing

- **Why normalized embeddings turn cosine similarity into a dot product**: cosine similarity divides the dot product by the product of both vectors' magnitudes. If both vectors are already unit-length, that denominator becomes 1 and the formula collapses to just the dot product — normalizing at encode time is what makes this simplification valid.
- **SHA-256 collision risk is not a practical concern here**: the probability of two different texts producing the same hash is astronomically small — "same hash" can be treated as "same content" without hedging.
- **Locality-Sensitive Hashing (LSH)**, including MinHash and SimHash, is the broader family of techniques that make near-duplicate detection scale past brute-force pairwise comparison — worth knowing by name even if current scale doesn't yet require it.

### Revision Summary

> Two-stage dedup, cheapest-and-most-certain first: a content hash catches exact duplicates for free across every document; cosine similarity (via normalized-embedding dot products) catches reworded near-duplicates, but only on documents that survived the hash check, because embeddings cost real money. Pairwise comparison is the scaling limit — approximate nearest neighbor indexes are the production answer past a few thousand documents. Flagging duplicates is not the same as resolving which copy to keep — that needs an explicit policy.

In [1]:
"""
dedup.py
---------
Two-stage duplicate detection for Documents coming from different filenames
(e.g. FD_Policy.json / FD_Policy_Final.json / FD_Policy_V2.json).

Stage 1 - HASH (cheap, exact):
    SHA-256 of the text. Identical content -> identical hash, regardless
    of filename. Runs on EVERY document; cost is negligible.

Stage 2 - COSINE SIMILARITY (expensive, near/fuzzy):
    Only runs on documents that SURVIVED stage 1. Embeddings cost real
    money/compute, so we never pay for them on content we'd already reject.
    Catches reworded duplicates: same fact, different phrasing.

Cost-ordering is the whole design: cheapest, most certain check first;
expensive, fuzzy check only on what's left.
"""

import hashlib
import numpy as np

# Loaded lazily so importing this module doesn't force a model download
# just to use content_hash() alone.
_model = None


def _get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    return _model


def content_hash(text: str) -> str:
    """A fingerprint for a piece of text. Identical text -> identical hash."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def find_duplicates(documents: list, near_duplicate_threshold: float = 0.97) -> list:
    """Returns a list of (doc_index, duplicate_of_index, 'exact' | 'near (score)')."""
    seen_hashes = {}          # hash -> index of the FIRST document with that hash
    kept_for_embedding = []   # (index, text) for docs that survived the hash check
    duplicates = []

    # ---- Stage 1: hash check (exact duplicates) ----
    for i, doc in enumerate(documents):
        h = content_hash(doc["page_content"])
        if h in seen_hashes:
            duplicates.append((i, seen_hashes[h], "exact"))
        else:
            seen_hashes[h] = i
            kept_for_embedding.append((i, doc["page_content"]))

    # ---- Stage 2: cosine similarity (near duplicates) ----
    if len(kept_for_embedding) > 1:
        indices, texts = zip(*kept_for_embedding)
        model = _get_model()
        embeddings = model.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)

        already_flagged = set()
        for a in range(len(indices)):
            if indices[a] in already_flagged:
                continue
            for b in range(a + 1, len(indices)):
                if indices[b] in already_flagged:
                    continue
                score = float(np.dot(embeddings[a], embeddings[b]))  # vectors normalized -> dot == cosine
                if score >= near_duplicate_threshold:
                    duplicates.append((indices[b], indices[a], f"near ({score:.3f})"))
                    already_flagged.add(indices[b])

    return duplicates


if __name__ == "__main__":
    from document import make_document

    test_docs = [
        make_document("FD rate is 7.1 percent for 24 month tenure.", {"source": "FD_Policy.json"}),
        make_document("FD rate is 7.1 percent for 24 month tenure.", {"source": "FD_Policy_Final.json"}),
        make_document("FD rate is 7.1 percent for a 24 month tenure period.", {"source": "FD_Policy_V2.json"}),
        make_document("Premature withdrawal incurs a 1 percent penalty.", {"source": "FD_Policy.json"}),
    ]
    dupes = find_duplicates(test_docs, near_duplicate_threshold=0.85)
    for idx, of_idx, kind in dupes:
        print(f"  Document {idx} ({test_docs[idx]['metadata']['source']}) is a {kind} "
              f"duplicate of Document {of_idx} ({test_docs[of_idx]['metadata']['source']})")


  Document 1 (FD_Policy_Final.json) is a exact duplicate of Document 0 (FD_Policy.json)
  Document 2 (FD_Policy_V2.json) is a near (0.993) duplicate of Document 0 (FD_Policy.json)
